In [36]:
from pyprojroot import here
from langchain_community.utilities import SQLDatabase



In [37]:
db_path=str(here("data"))+"/sqldb.db"
print(db_path)
db=f"sqlite:///{db_path}"
print(db)
sql_db=SQLDatabase.from_uri(db)
print(sql_db)

D:\ostad\AI Agent Development\Database interaction\data/sqldb.db
sqlite:///D:\ostad\AI Agent Development\Database interaction\data/sqldb.db


In [38]:
print(sql_db.dialect)

sqlite


In [39]:
print(sql_db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [40]:
sql_db._execute("SELECT * FROM Artist LIMIT 10")

[{'ArtistId': 1, 'Name': 'AC/DC'},
 {'ArtistId': 2, 'Name': 'Accept'},
 {'ArtistId': 3, 'Name': 'Aerosmith'},
 {'ArtistId': 4, 'Name': 'Alanis Morissette'},
 {'ArtistId': 5, 'Name': 'Alice In Chains'},
 {'ArtistId': 6, 'Name': 'Antônio Carlos Jobim'},
 {'ArtistId': 7, 'Name': 'Apocalyptica'},
 {'ArtistId': 8, 'Name': 'Audioslave'},
 {'ArtistId': 9, 'Name': 'BackBeat'},
 {'ArtistId': 10, 'Name': 'Billy Cobham'}]

## **Test the Access to the Environment Variable**

In [41]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [42]:
os.environ['GITHUB TOKEN']=""
token=os.environ.get("GITHUB TOKEN")
endpoint="https://models.github.ai/inference"
model_name = "openai/gpt-4.1-mini"

if not token:
    raise ValueError("Please set GITHUB TOKEN environment variable")

llm_model=ChatOpenAI(
    model_name=model_name,
    openai_api_base=endpoint,
    openai_api_key=token,
    temperature=0.8,
)

## **Test the GPT Model**

In [43]:
from langchain_core.messages import SystemMessage
from langchain_core.messages import HumanMessage

In [44]:
msg=llm_model.invoke(
    [HumanMessage(
        content="What is the capital of Bangladesh?"
    )]
)
print(msg.content)

The capital of Bangladesh is Dhaka.


## **Create SQL QUERY Chain**

In [45]:
from langchain_core.prompts import ChatPromptTemplate
#It creates a prompt template
from langchain_core.runnables import RunnablePassthrough
#It takes the input and pass it through unchanged
from langchain_core.output_parsers import StrOutputParser
#ite returns the main content from AI message

In [46]:
sql_prompt=ChatPromptTemplate.from_template(
    """ Given Database Schema below. Write the SQL Query from the user's question.
    Database Schema: {schema}
    user's Question: {question}
    SQL Query:"""
)
print(prompt)

<function prompt at 0x000001DFCF6B5260>


In [47]:
schema_info=sql_db.get_table_info()

In [48]:
sql_chain=RunnablePassthrough.assign(schema=lambda _:schema_info) | sql_prompt|llm_model| StrOutputParser()

In [49]:
response=sql_chain.invoke(
    {
        "question":"How many employees are there?",
    }
)

In [50]:
print(sql_chain)

first=RunnableAssign(mapper={
  schema: RunnableLambda(lambda _: schema_info)
}) middle=[ChatPromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template=" Given Database Schema below. Write the SQL Query from the user's question.\n    Database Schema: {schema}\n    user's Question: {question}\n    SQL Query:"), additional_kwargs={})]), ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E022579950>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E02257B610>, root_client=<openai.OpenAI object at 0x000001E022579A90>, root_async_client=<openai.AsyncOpenAI object at 0x000001E022579D10>, model_name='

In [51]:
print(response)

```sql
SELECT COUNT(*) AS NumberOfEmployees
FROM Employee;
```


## **Handling Different Types of response from AI output**

In [52]:
import re

In [75]:
def sql_query_handle(response):
    """
    Extract SQL Query from various response formats:
    1.Markdown Code Blocks: ```sql.....```
    2.SQLQuery: Label Format
    3.Plain SQL Query:
    """
    markdown_code_blocks=re.search(r'```sql\s*(.*?)\s*```',response,re.IGNORECASE|re.DOTALL)
    if markdown_code_blocks:
        sql_query=markdown_code_blocks.group(1).strip()
        sql_query = " ".join(markdown_code_blocks.group(1).split())
        return sql_query

    label_format=re.search(r"SQLQuery:\s*(.*?)(?:\n|$)",response,re.IGNORECASE)
    if label_format:
        sql_query=label_format.group(1).strip()
        return sql_query

    plain_query=re.match(r"^\s*(SELECT|DELETE|DROP|CREATE|INSERT|UPDATE|ALTER)\s+",response,re.IGNORECASE)
    if plain_query:
        sql_query=response.strip()
        return sql_query

    return None

## **Test the function with Different Format**

In [81]:
test_response=[
    #markdown_code_blocks
    """"```sql
    SELECT COUNT(*) AS NumberOfEmployees
    FROM Employee;```""",
    #Label Format
    "SQLQuery: SELECT * FROM Artist LIMIT 10;",
    #Plain SQL
    "SELECT COUNT(*) FROM Employee;"
]

In [82]:
print(type(test_response))
print(test_response)

<class 'list'>
['"```sql\n    SELECT COUNT(*) AS NumberOfEmployees\n    FROM Employee;```', 'SQLQuery: SELECT * FROM Artist LIMIT 10;', 'SELECT COUNT(*) FROM Employee;']


In [87]:
for i, test in enumerate(test_response,1):
    response_output=sql_query_handle(test)
    print(f"\nTest-{i}:")
    print(f"Input: {test[:50]}...")
    print(f"Output:{response_output}")



Test-1:
Input: "```sql
    SELECT COUNT(*) AS NumberOfEmployees
 ...
Output:SELECT COUNT(*) AS NumberOfEmployees FROM Employee;

Test-2:
Input: SQLQuery: SELECT * FROM Artist LIMIT 10;...
Output:SELECT * FROM Artist LIMIT 10;

Test-3:
Input: SELECT COUNT(*) FROM Employee;...
Output:SELECT COUNT(*) FROM Employee;


## **Put the Output from Chain Query to function**

In [84]:
response_output=sql_query_handle(response)
print(response_output)

SELECT COUNT(*) AS NumberOfEmployees FROM Employee;


In [85]:
sql_db.run(response_output)

'[(8,)]'